# SD-029 | Synthetic data generator

**Sprint 2 — Operations Analytics**

I generate five fully synthetic tables with a fixed seed. They describe a teaching example, not a real delivery operation.


## 1. Configure the generator

The seed, route settings, and outcome rules define the generated sample.


In [45]:
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 20250329
N_ROUTES = 48
N_RIDERS = 16
STOPS_PER_ROUTE = 10
START_DATE = "2025-05-01"
ZONES = ["Central", "North", "East", "West", "South", "Outer Ring"]
VEHICLE_TYPES = ["bicycle", "scooter", "van"]
DAILY_CAPACITIES = [10, 12, 14]
RAIN_OPTIONS_MM = [0, 0, 0, 2, 5, 10, 18]
SINGLE_FAILURE_PROBABILITY = 0.015
TABLE_NAMES = ("orders", "deliveries", "riders", "routes", "delivery_events")
LONG_DELAY_ORDER = "ORD-0001"
RETURNED_ORDER = "ORD-0480"
DATA_DIR = Path(__file__).resolve().parents[1] / "data" if "__file__" in globals() else Path("../data")
RANDOM = np.random.default_rng(SEED)

## 2. Create riders, routes, orders, deliveries, and event histories

Each route is followed by its stops so the seeded random draws happen in the same order on every run.


In [46]:
riders = pd.DataFrame([
    {
        "rider_id": f"R-{number:03d}",
        "experience_months": int(RANDOM.integers(2, 49)),
        "vehicle_type": RANDOM.choice(VEHICLE_TYPES),
        "daily_capacity": int(RANDOM.choice(DAILY_CAPACITIES)),
    }
    for number in range(1, N_RIDERS + 1)
])
experience_by_rider = riders.set_index("rider_id")["experience_months"].to_dict()
routes, orders, deliveries, events = [], [], [], []

In [47]:
def add_event(events, order_id, event_type, event_time, notes=""):
    events.append({
        "event_id": f"EV-{len(events) + 1:05d}",
        "order_id": order_id,
        "event_type": event_type,
        "event_time": event_time.strftime("%Y-%m-%d %H:%M:%S"),
        "notes": notes,
    })

In [48]:
def make_route(route_number):
    route_date = pd.Timestamp(START_DATE) + pd.Timedelta((route_number - 1) // 2, unit="D")
    route_start = route_date + pd.Timedelta(8 if route_number % 2 else 13, unit="h")
    zone = ZONES[int(RANDOM.integers(0, len(ZONES)))]
    traffic = round(float(RANDOM.uniform(0.75, 1.55)), 2)
    rain_mm = float(RANDOM.choice(RAIN_OPTIONS_MM))
    if zone == "Outer Ring":
        planned_distance = round(float(RANDOM.uniform(14, 24)), 1)
    else:
        planned_distance = round(float(RANDOM.uniform(7, 13)), 1)
    # Small random variation makes actual route distance a little longer.
    actual_distance = round(planned_distance * (1 + max(0, traffic - 1) * 0.08
                          + rain_mm / 150 + RANDOM.uniform(0.02, 0.12)), 1)
    route = {
        "route_id": f"RT-{route_number:03d}",
        "route_date": route_date.strftime("%Y-%m-%d"),
        "rider_id": f"R-{((route_number - 1) % N_RIDERS) + 1:03d}",
        "service_zone": zone, "traffic_index": traffic, "rain_mm": rain_mm,
        "planned_stops": STOPS_PER_ROUTE, "planned_distance_km": planned_distance,
        "actual_distance_km": actual_distance,
    }
    return route, route_start

## 3. Create orders, deliveries, and event histories

Each stop receives one delivery outcome and a sequence of event records.


In [49]:
def make_delivery_outcome(order_id, route, stop_sequence, promised_by):
    package_weight = round(float(RANDOM.uniform(0.3, 12.0)), 1)
    experience = experience_by_rider[route["rider_id"]]
    attempt_chance = min(0.24, 0.03 + (0.08 if route["service_zone"] == "Outer Ring" else 0)
                         + max(0, 12 - experience) * 0.009 + (0.03 if package_weight > 8 else 0))
    failed_attempts = int(RANDOM.random() < attempt_chance)
    if RANDOM.random() < SINGLE_FAILURE_PROBABILITY or order_id in (LONG_DELAY_ORDER, RETURNED_ORDER):
        failed_attempts = 2
    delay = int(round(7 + max(0, route["traffic_index"] - 1) * 35 + route["rain_mm"] * 0.8
                       + stop_sequence * 0.85 - experience * 0.2 + RANDOM.normal(0, 12)))
    delay = max(-15, min(delay, 75))
    if failed_attempts == 1:
        delay = max(delay, int(RANDOM.integers(55, 71)))
    elif failed_attempts == 2:
        delay = max(delay, int(RANDOM.integers(95, 126)))
    if order_id == LONG_DELAY_ORDER:
        delay = 185
    is_returned = order_id == RETURNED_ORDER
    delivered_at = pd.NaT if is_returned else promised_by + pd.Timedelta(delay, unit="m")
    if is_returned:
        status, reason = "returned_to_depot", "recipient_unavailable"
    elif order_id == LONG_DELAY_ORDER:
        status, reason = "delivered", "restricted_access"
    else:
        status, reason = "delivered", "recipient_unavailable" if failed_attempts else "none"
    return package_weight, failed_attempts, delay, is_returned, delivered_at, status, reason

In [50]:
def add_order_events(order_id, route_start, stop_sequence, promised_by, outcome):
    _, failed_attempts, _, is_returned, delivered_at, _, reason = outcome
    add_event(events, order_id, "order_placed", route_start - pd.Timedelta(2, unit="h"))
    add_event(events, order_id, "route_assigned", route_start - pd.Timedelta(15, unit="m"))
    add_event(events, order_id, "out_for_delivery",
              route_start + pd.Timedelta(6 * (stop_sequence - 1), unit="m"))
    for attempt in range(failed_attempts):
        attempt_time = promised_by + pd.Timedelta(5 + 40 * attempt, unit="m")
        add_event(events, order_id, "attempt_failed", attempt_time, reason)
        add_event(events, order_id, "reattempted", attempt_time + pd.Timedelta(25, unit="m"), "Returned to the route")
    if is_returned:
        add_event(events, order_id, "returned_to_depot", promised_by + pd.Timedelta(180, unit="m"),
                  "Recipient unavailable after two attempts")
    elif order_id == LONG_DELAY_ORDER:
        add_event(events, order_id, "delivered", delivered_at,
                  "Deliberate edge case: restricted-access delivery was 185 minutes late")
    else:
        add_event(events, order_id, "delivered", delivered_at)

In [51]:
def make_order(route_number, route, route_start, stop_sequence):
    order_id = f"ORD-{(route_number - 1) * STOPS_PER_ROUTE + stop_sequence:04d}"
    promised_by = route_start + pd.Timedelta(15 + 8 * stop_sequence, unit="m")
    outcome = make_delivery_outcome(order_id, route, stop_sequence, promised_by)
    package_weight, failed_attempts, delay, is_returned, delivered_at, status, reason = outcome
    orders.append({
        "order_id": order_id, "route_id": route["route_id"], "order_date": route["route_date"],
        "service_zone": route["service_zone"], "stop_sequence": stop_sequence,
        "ordered_at": (route_start - pd.Timedelta(2, unit="h")).strftime("%Y-%m-%d %H:%M:%S"),
        "promised_by": promised_by.strftime("%Y-%m-%d %H:%M:%S"),
        "parcel_weight_kg": package_weight, "promised_sla_hours": 4,
    })
    deliveries.append({
        "delivery_id": order_id.replace("ORD", "DEL"), "order_id": order_id,
        "route_id": route["route_id"], "rider_id": route["rider_id"],
        "dispatch_time": route_start.strftime("%Y-%m-%d %H:%M:%S"),
        "delivered_at": "" if is_returned else delivered_at.strftime("%Y-%m-%d %H:%M:%S"),
        "delivery_status": status, "delay_minutes": np.nan if is_returned else delay,
        "failed_attempts": failed_attempts, "exception_reason": reason,
    })
    add_order_events(order_id, route_start, stop_sequence, promised_by, outcome)

In [52]:
for route_number in range(1, N_ROUTES + 1):
    route, route_start = make_route(route_number)
    routes.append(route)
    for stop_sequence in range(1, STOPS_PER_ROUTE + 1):
        make_order(route_number, route, route_start, stop_sequence)

## 3. Assemble, validate, and export the five tables


In [53]:
tables = {
    "orders": pd.DataFrame(orders),
    "deliveries": pd.DataFrame(deliveries),
    "riders": riders,
    "routes": pd.DataFrame(routes),
    "delivery_events": pd.DataFrame(events),
}
assert tuple(tables) == TABLE_NAMES, "The generator must return exactly the five required menu tables."
assert tables["deliveries"].loc[0, "order_id"] == LONG_DELAY_ORDER
assert tables["deliveries"].loc[0, "delay_minutes"] == 185
assert tables["deliveries"].iloc[-1]["order_id"] == RETURNED_ORDER
assert tables["deliveries"].iloc[-1]["delivery_status"] == "returned_to_depot"

The seeded generator creates the same five named tables on every run.


In [54]:
DATA_DIR.mkdir(parents=True, exist_ok=True)
for name in TABLE_NAMES:
    tables[name].to_csv(DATA_DIR / f"{name}.csv", index=False)

The generated tables are ready for descriptive analysis.


In [55]:
pd.DataFrame({"table": TABLE_NAMES, "rows": [len(tables[name]) for name in TABLE_NAMES]})

,table,rows
0,orders,480
1,deliveries,480
2,riders,16
3,routes,48
4,delivery_events,2000
